# Compiler output → GPU execution (Tesla T4)
1. Runtime → Change runtime type → **T4 GPU**.
2. Locally, `scripts/package_for_colab.sh` makes `bert-cuda.zip` next to the repo. Upload it in the upload cell.
3. Run all cells. The main cell takes about 10–15 minutes (8 rebuilds + ~14 Nsight Compute runs).
4. The last cell downloads `perf_results.zip`; unzip it into `bert-cuda/results/perf/`.

The predictions being tested are in `perf/TEST_PLAN.md`.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version,clocks.max.sm,power.limit --format=csv
!nvcc --version | tail -2

In [ ]:
# Find Nsight Compute (ncu). Colab usually ships it with the CUDA toolkit but not always on PATH.
import os, glob, subprocess
cands = glob.glob('/usr/local/cuda*/bin/ncu') + glob.glob('/usr/local/cuda*/nsight-compute*/ncu') + glob.glob('/opt/nvidia/nsight-compute/*/ncu')
if cands:
    os.environ['PATH'] = os.path.dirname(cands[0]) + ':' + os.environ['PATH']
print('ncu:', subprocess.run('which ncu && ncu --version | tail -1', shell=True, capture_output=True, text=True).stdout or 'NOT FOUND')

In [ ]:
# If ncu was NOT FOUND above, uncomment and run this (installs Nsight Compute for the toolkit version):
# v = subprocess.run("nvcc --version | grep -o 'release [0-9]*\\.[0-9]*' | cut -d' ' -f2 | tr . -", shell=True, capture_output=True, text=True).stdout.strip()
# !apt-get -qq update && apt-get -qq install -y cuda-nsight-compute-{v}
# cands = glob.glob('/usr/local/cuda*/bin/ncu') + glob.glob('/opt/nvidia/nsight-compute/*/ncu'); os.environ['PATH'] = os.path.dirname(cands[0]) + ':' + os.environ['PATH']; print(cands)

In [ ]:
from google.colab import files
files.upload()  # choose bert-cuda.zip
!unzip -q -o bert-cuda.zip

In [ ]:
%cd /content/bert-cuda
cap = !nvidia-smi --query-gpu=compute_cap --format=csv,noheader
arch = 'sm_' + cap[0].strip().replace('.', '')
print('building for', arch)
!ARCH={arch} bash perf/run_perf.sh 2>&1 | tee perf_run.log

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/perf/SUMMARY.md').read().split('## Plots')[0]))
for p in sorted(glob.glob('results/perf/plots/*.png')):
    display(Image(p))

In [ ]:
# Look at the compiler output yourself: the reg64 main loop, PAD=1 vs PAD=4
!head -60 results/perf/sass_loops/default_reg64.txt
!echo ----------; head -60 results/perf/sass_loops/default_reg64_pad4.txt

In [ ]:
!cp perf_run.log results/perf/ && cd results && zip -qr ../perf_results.zip perf && cd .. && ls -lh perf_results.zip
files.download('perf_results.zip')